# Stage 05a: Initial Model Training

**Purpose:** Quick validation with default hyperparameters

**Inputs:**
- data/04c_train_encoded.parquet
- data/04c_test_encoded.parquet
- data/04c_train_base_margin.parquet
- data/04c_test_base_margin.parquet
- config_generated/04c_monotonicity_constraints.yaml

**Outputs:**
- models/05a_model_initial.json
- results/05a_predictions_train.parquet
- results/05a_predictions_test.parquet
- results/05a_metrics.yaml

In [1]:
config_path = "config/car_coll/v1"

In [2]:
# Parameters
config_path = "config/car_coll/v1"


In [3]:
import matplotlib
matplotlib.use("Agg")

import pandas as pd
import numpy as np
import xgboost as xgb
import yaml, os, sys
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error

sys.path.insert(0, str(Path.cwd() / "lib"))
from utils import setup_notebook_environment

print("########################################")
print("# STAGE 05a: INITIAL MODEL TRAINING")
print("########################################")

project_root = setup_notebook_environment()

########################################
# STAGE 05a: INITIAL MODEL TRAINING
########################################


In [4]:
print(f"Python: {sys.version}")
print(f"XGBoost: {xgb.__version__}")

Python: 3.11.15 (main, Jun 11 2026, 15:14:57) [Clang 20.1.8 ]
XGBoost: 3.1.1


In [5]:
# Get machine config
pc_num = open("current.pc").read().strip()
pc_id = f"PC{pc_num}"

config_file = f"{config_path}/config.yaml"
with open(config_file, "r") as f:
    cfg = yaml.safe_load(f)

output_base = cfg["machines"][pc_id]["paths"]["output_path"]
target = cfg["experiment"]["target"]
exposure = cfg["experiment"]["exposure"]

In [6]:
# Load encoded features (already filtered by exclusions in 04c)
print(f"\n* Loading encoded features...")
X_train = pd.read_parquet(f"{output_base}/data/04c_train_encoded.parquet")
X_test = pd.read_parquet(f"{output_base}/data/04c_test_encoded.parquet")

# Load original data for target/exposure
train_orig = pd.read_parquet(f"{output_base}/data/04b_train.parquet")
test_orig = pd.read_parquet(f"{output_base}/data/04b_test.parquet")

y_train = train_orig[target]
y_test = test_orig[target]
w_train = train_orig[exposure]
w_test = test_orig[exposure]

print(f"  Train: {X_train.shape}")
print(f"  Test: {X_test.shape}")
print(f"  Features: {X_train.shape[1]}")


* Loading encoded features...


  Train: (3757142, 196)
  Test: (3750266, 196)
  Features: 196


In [7]:
# Load base_margin if available
base_margin_file_train = f"{output_base}/data/04c_train_base_margin.parquet"
base_margin_file_test = f"{output_base}/data/04c_test_base_margin.parquet"

if os.path.exists(base_margin_file_train):
    print(f"\n* Loading base_margin...")
    base_margin_train = pd.read_parquet(base_margin_file_train)["base_margin"].values
    base_margin_test = pd.read_parquet(base_margin_file_test)["base_margin"].values
    print(f"  Train base_margin: mean={base_margin_train.mean():.4f}")
    print(f"  Test base_margin: mean={base_margin_test.mean():.4f}")
else:
    base_margin_train = None
    base_margin_test = None
    print(f"\n* No base_margin found (GLM init disabled)")


* Loading base_margin...


  Train base_margin: mean=nan
  Test base_margin: mean=nan


In [8]:
# Load monotonicity constraints
mono_file = f"{output_base}/config_generated/04c_monotonicity_constraints.yaml"
if os.path.exists(mono_file):
    print(f"\n* Loading monotonicity constraints...")
    with open(mono_file, "r") as f:
        mono_dict = yaml.safe_load(f)
    # Build constraints tuple in feature order
    feature_names = X_train.columns.tolist()
    monotone_constraints = tuple(mono_dict.get(f, 0) for f in feature_names)
    n_constrained = sum(1 for c in monotone_constraints if c != 0)
    print(f"  Loaded {n_constrained} constraints")
else:
    monotone_constraints = None
    print(f"\n* No monotonicity constraints found")


* Loading monotonicity constraints...
  Loaded 14 constraints


In [9]:
# Build XGBoost parameters
print(f"\n* Building XGBoost parameters...")
xgb_params = cfg["xgboost"].copy()
n_estimators = xgb_params.pop("n_estimators")

# Add monotonicity constraints if available
if monotone_constraints:
    xgb_params["monotone_constraints"] = monotone_constraints

# Fix eval_metric for tweedie
if "eval_metric" in xgb_params and "tweedie" in xgb_params["eval_metric"]:
    if "@" not in xgb_params["eval_metric"]:
        variance_power = xgb_params["tweedie_variance_power"]
        xgb_params["eval_metric"] = f"{xgb_params['eval_metric']}@{variance_power}"

print(f"  n_estimators: {n_estimators}")
print(f"  monotone_constraints: {monotone_constraints is not None}")


* Building XGBoost parameters...
  n_estimators: 500
  monotone_constraints: True


In [10]:
# Train model
print(f"\n* Training XGBoost model...")

model = xgb.XGBRegressor(n_estimators=n_estimators, **xgb_params)

model.fit(
    X_train, y_train,
    sample_weight=w_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    sample_weight_eval_set=[w_train, w_test],
    verbose=100
)

print(f"\n* Training complete")
print(f"  Best iteration: {model.best_iteration}")


* Training XGBoost model...


[0]	validation_0-tweedie-nloglik@1.5:57.67802	validation_1-tweedie-nloglik@1.5:58.42930


[100]	validation_0-tweedie-nloglik@1.5:55.13699	validation_1-tweedie-nloglik@1.5:56.25770


[200]	validation_0-tweedie-nloglik@1.5:54.73443	validation_1-tweedie-nloglik@1.5:56.16608


[300]	validation_0-tweedie-nloglik@1.5:54.43335	validation_1-tweedie-nloglik@1.5:56.14693


[392]	validation_0-tweedie-nloglik@1.5:54.17314	validation_1-tweedie-nloglik@1.5:56.14848



* Training complete
  Best iteration: 342


In [11]:
# Generate predictions
print(f"\n* Generating predictions...")
pred_train = model.predict(X_train)
pred_test = model.predict(X_test)

# Calculate metrics
mae_train = mean_absolute_error(y_train, pred_train, sample_weight=w_train)
mae_test = mean_absolute_error(y_test, pred_test, sample_weight=w_test)
rmse_train = np.sqrt(mean_squared_error(y_train, pred_train, sample_weight=w_train))
rmse_test = np.sqrt(mean_squared_error(y_test, pred_test, sample_weight=w_test))

print(f"\n* Metrics:")
print(f"  Train MAE: {mae_train:.4f}, RMSE: {rmse_train:.4f}")
print(f"  Test MAE: {mae_test:.4f}, RMSE: {rmse_test:.4f}")


* Generating predictions...



* Metrics:
  Train MAE: 395.8659, RMSE: 2404.3867
  Test MAE: 401.5543, RMSE: 2466.5983


In [12]:
# Save model
os.makedirs(f"{output_base}/models", exist_ok=True)
model_file = f"{output_base}/models/05a_model_initial.json"
model.get_booster().save_model(model_file)
print(f"\n* Saved model: {model_file}")


* Saved model: output/car_coll/v1/models/05a_model_initial.json


In [13]:
# Save predictions
os.makedirs(f"{output_base}/results", exist_ok=True)

pd.DataFrame({
    "actual": y_train,
    "pred": pred_train,
    "exposure": w_train
}).to_parquet(f"{output_base}/results/05a_predictions_train.parquet", index=True)

pd.DataFrame({
    "actual": y_test,
    "pred": pred_test,
    "exposure": w_test
}).to_parquet(f"{output_base}/results/05a_predictions_test.parquet", index=True)

print(f"* Saved predictions")

* Saved predictions


In [14]:
# Save metrics
metrics = {
    "stage": "05a_initial",
    "train": {"mae": float(mae_train), "rmse": float(rmse_train)},
    "test": {"mae": float(mae_test), "rmse": float(rmse_test)},
    "best_iteration": int(model.best_iteration),
    "n_features": int(X_train.shape[1]),
    "monotone_constraints_applied": monotone_constraints is not None,
    "base_margin_applied": base_margin_train is not None
}

with open(f"{output_base}/results/05a_metrics.yaml", "w") as f:
    yaml.dump(metrics, f)

print(f"* Saved metrics")

* Saved metrics


In [15]:
print("\n########################################")
print("# STAGE 05a: COMPLETE")
print("########################################")


########################################
# STAGE 05a: COMPLETE
########################################
